<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Python for Algorithmic Trading

&copy; Dr. Yves J. Hilpisch | The Python Quants GmbH

http://tpq.io | [training@tpq.io](mailto:trainin@tpq.io) | [@dyjh](http://twitter.com/dyjh)

## Event-Based Backtesting

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_practice.git
import sys
sys.path.append('python_for_algo_trading_practice')


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from pylab import plt
plt.style.use('seaborn-v0_8')
%config InlineBackend.figure_format = 'svg'

## Event-Based View

... moving bar-by-bar through the historical data.

In [ ]:
url = 'https://certificate.tpq.io/findata.csv'

In [ ]:
data = pd.read_csv(url, index_col=0, parse_dates=True).dropna()

In [ ]:
data['.SPX'].head()

In [ ]:
import time

In [ ]:
for bar in range(10):
    print(bar, data.index[bar], data['.SPX'].iloc[bar])
    time.sleep(0.25)

## `FinancialData` Class

In [ ]:
class FinancialData:
    url = 'https://certificate.tpq.io/findata.csv'
    def __init__(self, symbol):
        self.symbol = symbol
        self.retrieve_data()
        self.prepare_data()
    def retrieve_data(self):
        self.raw = pd.read_csv(self.url, index_col=0, parse_dates=True)
    def prepare_data(self):
        self.data = pd.DataFrame(self.raw[self.symbol]).dropna()
        self.data['r'] = np.log(self.data / self.data.shift(1))
    def plot_data(self, cols=None):
        if cols is None:
            cols = [self.symbol]
        self.data[cols].plot(title=self.symbol);

In [ ]:
fd = FinancialData('.SPX')

In [ ]:
fd.data.head()

In [ ]:
fd.plot_data()

## `BacktestingBase` Class

We are going to implement a **base class** for event-based backtesting with:

* `__init__`
* `retrieve_data` (`FinancialData`)
* `prepare_data` (`FinancialData`)
* `plot_data` (`FinancialData`)
* `get_date_price`
* `print_balance`
* `print_net_wealth`
* `place_buy_order`
* `place_sell_order`
* `close_out`

In [ ]:
class BacktestingBase(FinancialData):
    def __init__(self, symbol, amount, ptc):
        super().__init__(symbol)
        self.initial_amount = amount
        self.current_balance = amount
        self.ptc = ptc
        self.units = 0
        self.trades = 0
    def get_date_price(self, bar):
        date = str(self.data.index[bar])[:10]
        price = self.data[self.symbol].iloc[bar]
        return date, price
    def print_balance(self, bar):
        date, price = self.get_date_price(bar)
        print(f'{date} | current balance = {self.current_balance:.2f}')
    def print_net_wealth(self, bar):
        date, price = self.get_date_price(bar)
        net_wealth = self.current_balance + self.units * price
        print(f'{date} | net wealth = {net_wealth:.2f}')
    def place_buy_order(self, bar, units=None, amount=None):
        date, price = self.get_date_price(bar)
        if units is None:
            units = int(amount / (price * (1 + self.ptc)))
        self.units += units
        self.current_balance -= units * price * (1 + self.ptc)
        self.trades += 1
        print(f'{date} | bought {units} units for {price}')
        self.print_balance(bar)
        self.print_net_wealth(bar)
    def place_sell_order(self, bar, units=None, amount=None):
        date, price = self.get_date_price(bar)
        if units is None:
            units = int(amount / (price * (1 + self.ptc)))
        self.units -= units
        self.current_balance += units * price * (1 - self.ptc)
        self.trades += 1
        print(f'{date} | sold {units} units for {price}')
        self.print_balance(bar)
        self.print_net_wealth(bar)
    def close_out(self, bar):
        date, price = self.get_date_price(bar)
        print(55 * '=')
        print(f'{date} | *** CLOSING OUT POSITION ***')
        print(55 * '=')
        print(f'{date} | closing {self.units} units for {price}')
        self.current_balance += self.units * price
        self.units = 0
        self.trades += 1
        self.print_balance(bar)
        perf = (self.current_balance / self.initial_amount - 1) * 100
        print(f'{date} | performance [%] = {perf:.3f}')
        print(f'{date} | trades [#] = {self.trades}')

In [ ]:
bb = BacktestingBase('.SPX', 10000, ptc=0.01)

In [ ]:
bb.get_date_price(100)

In [ ]:
bb.print_balance(100)

In [ ]:
bb.place_buy_order(500, units=4)

In [ ]:
bb.print_balance(1000)

In [ ]:
bb.print_net_wealth(1000)

In [ ]:
bb.place_sell_order(1100, units=1)

In [ ]:
bb.place_sell_order(1300, amount=2800)

In [ ]:
bb.units

In [ ]:
bb.close_out(1750)

## `SMABacktester` Class

In [ ]:
class SMABacktester(BacktestingBase):
    def prepare_statistics(self):
        self.data['SMA1'] = self.data[self.symbol].rolling(self.SMA1).mean()
        self.data['SMA2'] = self.data[self.symbol].rolling(self.SMA2).mean()
    def backtest_strategy(self, SMA1, SMA2, units):
        self.SMA1 = SMA1
        self.SMA2 = SMA2
        self.prepare_statistics()
        print(55 * '=')
        print(f'*** START BACKTESTING ***')
        print(f'SYM={self.symbol} | SMA1={SMA1} | SMA2={SMA2}')
        print(55 * '=')
        self.units = 0
        self.trades = 0
        self.position = 'neutral'
        self.current_balance = self.initial_amount
        for bar in range(self.SMA2, len(self.data)):
            date, price = self.get_date_price(bar)
            if self.data['SMA1'].iloc[bar] > self.data['SMA2'].iloc[bar]:
                signal = 'long'
            else:
                signal = 'short'
            if self.position in ['neutral', 'short'] and signal == 'long':
                # going long
                if self.position == 'short':
                    self.place_buy_order(bar, units=units)
                self.place_buy_order(bar, units=units)
                self.position = 'long'
                print(55 * '=')
            elif self.position in ['neutral', 'long'] and signal == 'short':
                # going short
                if self.position == 'long':
                    self.place_sell_order(bar, units=units)
                self.place_sell_order(bar, units=units)
                self.position = 'short'
                print(55 * '=')
        self.close_out(bar)

In [ ]:
sma = SMABacktester('EUR=', 10000, ptc=0.01)

In [ ]:
sma.backtest_strategy(42, 252, 5000)

In [ ]:
sma.data.tail()

In [ ]:
sma.plot_data([sma.symbol, 'SMA1', 'SMA2'])

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>